# 30. 코드리뷰용 구조와 실험 기록 확인

`code_review_guide.md`를 먼저 읽고 아래 셀을 순서대로 실행하세요. 저장된 설정·요약 파일만 읽습니다. DB 조회·임베딩·생성 API·파일 저장은 없습니다. 과거 실험 숫자를 새 실행 결과로 바꾸지 않습니다.


In [ ]:
# [프로젝트 추가] 어느 폴더에서 실행해도 현재 프로젝트의 src 폴더를 찾습니다.
from pathlib import Path
import json
import sys

후보_폴더 = [Path.cwd(), *Path.cwd().parents]
프로젝트_폴더 = next((폴더 for 폴더 in 후보_폴더
    if (폴더 / 'src/car_search_rag/zzong_santafe_lag/source_profile.py').exists()), None)
if 프로젝트_폴더 is None:
    raise FileNotFoundError('프로젝트 안에서 노트북을 열어 주세요.')
소스_폴더 = str(프로젝트_폴더 / 'src')
if 소스_폴더 not in sys.path:
    sys.path.insert(0, 소스_폴더)
개인_폴더 = 프로젝트_폴더 / 'src/car_search_rag/zzong_santafe_lag'
print('현재 확인할 개인 코드:', 개인_폴더)


## 1. 부모·자식·그림의 개수

아래 값은 현재 승인한 저장 버전의 기준입니다. 실제 원격 DB 개수를 지금 조회하는 셀이 아닙니다. 이 숫자만 바꿔 새 자료를 통과시키지 않습니다.


In [ ]:
# [프로젝트 추가] source_profile은 공개 기준만 담으며 모델·DB 연결을 준비하지 않습니다.
from car_search_rag.zzong_santafe_lag.source_profile import FULL_SOURCE

print('PDF 쪽수:', FULL_SOURCE.page_count)
print('부모 묶음:', FULL_SOURCE.parent_count)
print('검색용 자식 청크:', FULL_SOURCE.chunk_count)
print('그림 파일:', FULL_SOURCE.image_count)
print('원문 저장 작업 ID:', FULL_SOURCE.run_id)


## 2. 로컬/OpenAI의 같은 30문항 비교

Hit@5는 상위 5개 안에서 기대 근거를 찾은 질문 비율입니다. MRR@5는 첫 기대 근거가 앞에 나올수록 높습니다. 정답 ID 기준 보조 Precision은 새 내용 판정 Precision과 구별하세요.


In [ ]:
# [프로젝트 적용] 전환 당시 저장한 비교 요약을 읽습니다. 새 API 요청은 없습니다.
비교_파일 = 프로젝트_폴더 / 'data/zzong_santafe_lag/openai_embeddings/comparison_summary.json'
if 비교_파일.exists():
    비교 = json.loads(비교_파일.read_text(encoding='utf-8'))
    print('질문 수:', 비교['question_count'])
    for 모델, 지표 in 비교['metrics'].items():
        print(모델, 지표)
    print('이 실험의 한계:', 비교['limitations'])
else:
    print('이 PC에 당시 비교 요약이 없습니다. code_review_guide.md의 기록을 참고하세요.')


## 3. 새 질문 평가와 답변 보완 구별하기

검색 후보의 관련성과 답변에 사용할 근거 선택은 서로 다른 단계입니다. 아래 숫자는 Codex 판정의 잠정값이고 사람 교차 검토는 완료하지 않았습니다.


In [ ]:
# [프로젝트 적용] 새 질문 평가와 이후 보완 요약을 읽어 당시 결과를 비교합니다.
새평가 = json.loads((개인_폴더 / 'fresh_evaluation_summary_20261005.json').read_text(encoding='utf-8'))
보완 = json.loads((개인_폴더 / 'answer_fix_summary_20261005.json').read_text(encoding='utf-8'))
print('새 질문 수:', 새평가['question_count'])
print('검색 관련성 지표:', 새평가['metrics'])
print('사람 교차 검토 완료:', 새평가['human_confirmed'])
print('선택한 직접 근거, 변경 전:', 보완['selection_direct_support_before'])
print('선택한 직접 근거, 변경 후:', 보완['selection_direct_support_after'])
print('검색 Precision 변경 여부:', not 보완['retrieval_precision_unchanged'])
print('대표 3문항의 최종 생성 상태:', 보완['final_generation_statuses'])
print('기존 동작 조건 일치:', 보완['regression_expectation_match'])
print('전체 챗봇 의미 정확도 100%를 뜻하는 결과는 아닙니다.')


## 코드리뷰 설명 순서

1. PDF 자료 준비와 사용자 질문 처리를 구별합니다.
2. 자식 청크로 찾고 부모 글·필수 각주·검토된 그림으로 확인한다고 설명합니다.
3. 임베딩과 TF-IDF를 함께 사용하는 자체 검색 코드임을 설명합니다.
4. OpenAI 임베딩 모델과 생성 모델의 역할을 구별합니다.
5. 전체 업로드 완료와 원문·그림 설명 검토 완료를 구별합니다.
6. 이번 중복 코드 정리 후 실제 검색·생성은 재실행하지 않았다고 설명합니다.

세부 파일 역할·하드코딩 처리·1분 발표문은 `code_review_guide.md`에 있습니다.
